# Ollama 세팅

## Ollama 설치

In [1]:
# 1. zstd 설치 (에러 해결용)
!sudo apt-get update && sudo apt-get install -y zstd

# 2. Ollama 재설치 시도
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,388 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntu

## Ollama 서버 실행

In [2]:
import subprocess
import time

# 1. Ollama 서버를 백그라운드에서 실행
# stdout/stderr를 무시하지 않고 연결해두면 나중에 로그 확인이 가능합니다.
process = subprocess.Popen(['ollama', 'serve'],
                           stdout=subprocess.PIPE,
                           stderr=subprocess.PIPE,
                           text=True)

# 2. 서버가 완전히 준비될 때까지 대기
print("Ollama 서버를 시작하는 중...")
time.sleep(10)
print("Ollama 서버가 준비되었습니다!")

Ollama 서버를 시작하는 중...
Ollama 서버가 준비되었습니다!


## Ollama 모델 다운로드

In [3]:
!ollama pull llama3

# RAG 구축

- 원석(PDF) 채굴 → 가공(Chunking) → 분류표 부착(Metadata) → 창고 적재(Vector DB)

- 청킹 코드 (sliding_window_chunking):

역할: 수천 페이지의 책을 읽기 좋게 핵심 문장 단위로 쪼개서 서가에 정리하는 과정이야.

상태: 이 작업이 끝나야 **vectorstore(도서관)**라는 데이터베이스가 만들어져.

- 검색 단계 (similarity_search):

역할: 사용자가 질문을 하면, 도서관(vectorstore)에서 네가 쪼개놓은 청크들 중 가장 관련 있는 **4개의 조각(Context)**을 뽑아오는 사서 역할이야.

- 생성 함수 (test_ollama):

역할: 사서가 가져온 4개의 조각을 읽고, 사용자가 이해하기 쉽게 최종 답변을 작성하는 작가 역할이야.

## 라이브러리 설치

In [4]:
!pip install langchain-core
!pip install -U langchain langchain-community langchain-huggingface langchain-text-splitters pymupdf sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/10

## RAG 청킹함수 설정

In [7]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings # 필수: 숫자 변환기
from langchain_community.vectorstores import Chroma      # 필수: 저장소
import re
from langchain_core.documents import Document

# 1. 임베딩 모델 설정 (이건 꼭 있어야 해!)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. 네가 만든 똑똑한 청킹 함수 (기존 splitter 대체)
def sliding_window_chunking(docs, window_size=5, step=2):
    """문장 단위로 슬라이딩 윈도우 청킹 수행 (JCCR 논문 전략 반영)"""
    chunked_docs = []

    for doc in docs:
        # 1. 문장 단위로 분리 (마침표, 물음표 등 기준)
        sentences = re.split(r'(?<=[.!?])\s+', doc.page_content)

        # 2. 슬라이딩 윈도우 적용
        for i in range(0, len(sentences), step):
            window = sentences[i : i + window_size]
            if not window: continue

            chunk_content = " ".join(window)

            # 메타데이터 유지 및 청크 정보 추가
            new_metadata = doc.metadata.copy()
            new_metadata["chunk_index"] = i // step

            chunked_docs.append(Document(page_content=chunk_content, metadata=new_metadata))

            # 마지막 문단 처리
            if i + window_size >= len(sentences): break

    return chunked_docs

print("✅ RAG 핵심 엔진(임베딩)과 커스텀 청킹 함수 준비 완료!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ RAG 핵심 엔진(임베딩)과 커스텀 청킹 함수 준비 완료!


## IPCC 문서 청킹 (all_processed_chunks 설정)

In [10]:
# 파일 로드 및 청킹 통합 실행
all_processed_chunks = []

# Google Drive 내 IPCC_data 폴더 경로
BASE_PATH = "/content/drive/MyDrive/IPCC_data/"

# 파일 목록을 Google Drive의 실제 파일명으로 업데이트합니다.
# (사용자 제공 예시를 바탕으로 다른 파일명 추정)
ipcc_files = [
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Section_1.pdf", # '세션1.pdf'에 해당한다고 추정
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Section_2.pdf",    # '세션2.pdf'에 해당한다고 추정
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Section_3.pdf",    # '세션3.pdf'에 해당한다고 추정
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Section_4.pdf",    # '세션4.pdf'에 해당한다고 추정
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Annex_1_Glossary.pdf", # '부속서_용어집.pdf'에 해당 (사용자 제공)
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Annex_2_Acronyms.pdf"  # '부속서_약어.pdf'에 해당한다고 추정
]

for file_name in ipcc_files:
    if os.path.exists(file_name):
        loader = PyMuPDFLoader(file_name)
        pages = loader.load()

        # 각 파일의 메타데이터 주입
        # 파일명에서 특정 접두사와 .pdf 확장자를 제거하여 섹션 이름을 추출합니다.
        section_name = os.path.basename(file_name).replace("IPCC_AR6_SYR_FullVolume_", "").replace(".pdf", "")
        for p in pages: p.metadata["section"] = section_name

        # 논문 기반 슬라이딩 윈도우 청킹 적용
        chunks = sliding_window_chunking(pages)
        all_processed_chunks.extend(chunks)
        print(f"✅ {section_name}: {len(chunks)}개 청크 생성 완료")
    else:
        print(f"❌ 파일이 존재하지 않습니다: {file_name}. Google Drive에 파일이 올바르게 업로드되었는지 확인해주세요.")

# 벡터 DB 생성
if all_processed_chunks:
    vectorstore = Chroma.from_documents(documents=all_processed_chunks, embedding=embeddings)
    print("🔥 모든 파일이 벡터 데이터베이스에 저장되었습니다!")
else:
    print("⚠️ 처리된 청크가 없어 벡터 데이터베이스를 생성할 수 없습니다.")

✅ Section_1: 18개 청크 생성 완료
✅ Section_2: 250개 청크 생성 완료
✅ Section_3: 209개 청크 생성 완료
✅ Section_4: 246개 청크 생성 완료
✅ Annex_1_Glossary: 176개 청크 생성 완료
✅ Annex_2_Acronyms: 8개 청크 생성 완료
🔥 모든 파일이 벡터 데이터베이스에 저장되었습니다!


## RQG 체인 완성

### 생성함수 생성

In [11]:
def test_ollama(prompt):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": "llama3",
        "prompt": prompt,
        "stream": False # 결과를 한 번에 받기 위해 False 설정
    }

    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()
        result = response.json()
        return result['response']
    except Exception as e:
        return f"에러 발생: {e}"

### 검색

In [13]:
import requests
import json

def ipcc_ai_analyst(question):
    # 1. 지식 검색 (상위 4개 조각 추출)
    docs = vectorstore.similarity_search(question, k=4)

    # 2. 컨텍스트 구성 및 출처 정리
    ## 찾아온 4개의 조각을 하나의 긴 텍스트로 합쳐. 이때 각 조각이 어느 섹션(Section 1, Annex 등)에서 왔는지 출처를 딱 붙여주지.
    context = "\n\n".join([f"[Source: {d.metadata['section']}]\n{d.page_content}" for d in docs])
    sources = list(set([d.metadata['section'] for d in docs]))

    # 3. IPCC 전용 프롬프트 설계 (JCCR 논문 기반 페르소나 적용)
    prompt = f"""
너는 IPCC 6차 보고서(AR6) 전문 분석가야. 아래 제공된 [Context]를 바탕으로 질문에 답해줘.

[Context]
{context}

[Question]
{question}

[Guidelines]
- 반드시 제공된 Context의 내용을 근거로 답변할 것.
- 답변은 전문적이고 신뢰감 있는 한국어로 작성할 것.
- 인류의 활동에 의한 기후 변화의 책임을 명확히 전달할 것 (Action-oriented).
- 전문 용어는 영어 원문을 괄호 안에 병기할 것 (예: 탄소 중립(Net Zero)).
- 답변 마지막에 참고한 섹션들을 나열할 것.

답변:
"""

    # 4. Ollama 호출 (우리가 이전에 만든 test_ollama 함수 사용)
    response = test_ollama(prompt)

    print(f"\n📢 분석 결과 (참조: {', '.join(sources)})")
    print("-" * 50)
    print(response)

# 🔥 드디어 실행!
ipcc_ai_analyst("탄소 중립(Net Zero)의 정의와 이를 달성하기 위해 필요한 정책적 노력은 무엇인가요?")


📢 분석 결과 (참조: Section_2, Annex_1_Glossary, Section_3)
--------------------------------------------------
🌎️💡

According to the provided context, the concept of net zero emissions can be applied at global or sub-global scales (e.g., regional, national and sub-national). At a global scale, the terms GHG neutrality and net zero GHG emissions are equivalent.

To achieve net zero emissions, it is essential to have policies, institutions, and milestones against which to track progress. The adoption and implementation of net zero emission targets by countries and regions depend on equity and capacity considerations.

Furthermore, the quantification of net zero emissions depends on the GHG emission metric chosen to compare emissions and removals of different gases, as well as the time horizon chosen for that metric.

To achieve net zero emissions, the following policy efforts are necessary:

1. Establishing a clear scope and plan of action for net zero emissions.
2. Formulating fair and equita

# 챗봇 구현

## 라이브러리 설치

In [14]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 109.8 MB/s eta 0:00:00


## app.py 생성

In [16]:
# 이전 인덱싱 셀에서 생성한 vectorstore 객체가 있는 상태에서 실행
vectorstore = Chroma.from_documents(
    documents=all_processed_chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"  # 이 경로를 지정해야 파일로 저장됩니다.
)
# vectorstore.persist() # 최신 버전은 자동 저장되지만 명시적으로 호출해도 좋습니다.
print("✅ 벡터 DB가 ./chroma_db 폴더에 물리적으로 저장되었습니다.")

✅ 벡터 DB가 ./chroma_db 폴더에 물리적으로 저장되었습니다.


In [15]:
%%writefile app.py
import streamlit as st
import requests
import os
import gc
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# --- 페이지 설정 ---
st.set_page_config(page_title="기후 위기 알리미: IPCC AR6 해설봇", page_icon="🌍", layout="wide")

# 메모리 정리 함수
def clear_memory():
    gc.collect()

# --- 리소스 로드 ---
@st.cache_resource
def load_resources():
    clear_memory()
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    db_path = "./chroma_db"
    if os.path.exists(db_path):
        vectorstore = Chroma(persist_directory=db_path, embedding_function=embeddings)
        return vectorstore
    else:
        st.error("❌ ./chroma_db 폴더를 찾을 수 없습니다.")
        return None

vectorstore = load_resources()

# --- Ollama API 호출 함수 ---
def ask_ollama(prompt):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": "llama3",
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": 800, # 답변 길이를 제한하여 리소스 절약 및 간결함 유지
            "temperature": 0.5   # 더 정확하고 일관된 답변을 위해 온도를 낮춤
        }
    }
    try:
        response = requests.post(url, json=payload)
        return response.json()['response']
    except Exception as e:
        return f"⚠️ Ollama 연결 에러 (서버 상태를 확인하세요)"

# --- UI 레이아웃 ---
st.title("🌍 기후 위기 알리미: IPCC 보고서 요약봇")
st.info("IPCC AR6 보고서의 핵심을 한국어 해설과 영어 원문으로 짧게 요약해 드립니다.")

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

if prompt := st.chat_input("질문을 입력하세요 (예: 해수면 상승의 원인은?)"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    if vectorstore:
        with st.chat_message("assistant"):
            with st.spinner("핵심 내용 요약 중..."):
                docs = vectorstore.similarity_search(prompt, k=3) # 검색 결과도 3개로 압축
                context = "\n\n".join([f"[Source: {d.metadata.get('section', 'AR6')}]\n{d.page_content}" for d in docs])
                sources = list(set([d.metadata.get('section', '보고서 원문') for d in docs]))

                # --- 숏폼 & 듀얼 언어 프롬프트 ---
                full_prompt = f"""너는 기후 변화의 역사를 이야기로 들려주는 'IPCC 기후 타임머신'이야.
[Context]의 과학적 사실을 바탕으로 질문에 답하되, 반드시 [과거-현재-미래] 순서로 한국어와 영어를 병기해줘.

[Rules]
1. Headline: 이모지와 함께 핵심 결론을 한 줄로 적을 것.
2. [Summary (KR)]:
   - (과거) 우리가 해온 행동과 원인 (비유 포함)
   - (현재) 지금 우리 눈앞에 벌어지는 현상
   - (미래) 앞으로 우리가 마주할 변화와 경고
3. [Original (EN)]: 위 한국어 내용을 대응하는 간결한 영어 요약 (Past-Present-Future 순서 유지).
4. [Action]: 미래를 바꾸기 위해 지금 당장 할 수 있는 일.
5. Reliability: 과학적 확신도를 별점(★)으로 표시.

[Context]
{context}

[Question]
{prompt}

Answer:"""

                response = ask_ollama(full_prompt)
                clear_memory()

                final_answer = f"{response}\n\n---\n**📚 참조:** {', '.join(sources)} 🌏"
                st.markdown(final_answer)
                st.session_state.messages.append({"role": "assistant", "content": final_answer})

Writing app.py


## 라이브러리 설치

In [17]:
# 2. 외부 노출용 localtunnel 설치
!npm install -g localtunnel

# 1. Cloudflared 설치
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 3s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇--2026-03-03 06:08:11--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.2.0/cloudflared-linux-amd64.deb [following]
--2026-03-03 06:08:11--  https://github.com/cloudflare/cloudflared/releases/download/2026.2.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/4fddf4d7-e02d-44dc-9e5a-ef9e28afdd54?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-03T06%3A46%3A45Z&rscd=attachment%3B+filename%3Dcloudflared-linux-am

## 서버 실행

In [18]:
import subprocess
import time

# 로그를 직접 출력하기 위해 쉘 명령어로 실행
# --logger.level=debug 설정을 추가하여 상세 로그를 확인합니다.
print("🚀 Streamlit 실행 중...")
with open("logs.txt", "w") as f:
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--logger.level=debug"], stdout=f, stderr=f)

time.sleep(5)
print("🔗 Cloudflare 터널링 시작...")
!cloudflared tunnel --url http://localhost:8501

🚀 Streamlit 실행 중...
🔗 Cloudflare 터널링 시작...
2026-03-03T06:08:28Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-03T06:08:28Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-03T06:08:34Z INF +--------------------------------------------------------------------------------------------+
2026-03-03T06:08:34Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-03T06:08:34Z INF |  https://in